# 08 -- Backtest Orchestration

> **Data mode: SYNTHETIC FIXTURE DATA.** Every filing/price/universe/sector record in this notebook comes from `_fixtures.py`, hand-authored for demonstration only. **Nothing computed in this notebook is a genuine historical result, and no cell here may be cited as evidence of reproducing `report_current.html`.** See `docs/reproducibility_findings.md` for the actual reproducibility status.

Runs the full top-level `run_filing_momentum_production_backtest` over the synthetic bundle. This is the same orchestration function `atlas-quant filing-momentum run-backtest` calls -- nothing in this notebook reimplements any part of it.

In [ ]:
import atlas_quant
print("atlas_quant path:", atlas_quant.__file__)

import sys
sys.path.insert(0, ".")  # this notebook's own directory (for _fixtures.py) -- not a legacy path hack
import _fixtures


In [ ]:
from atlas_quant.backtest.filing_momentum_runner import FilingMomentumBacktestConfig
from atlas_quant.strategies.filing_momentum_ml.production.orchestration import (
    ProductionRunInputs, run_filing_momentum_production_backtest,
)

bundle = _fixtures.build_synthetic_bundle()
calendar = _fixtures.build_trading_calendar(bundle)
periods = _fixtures.build_periods()
manifest = _fixtures.build_synthetic_manifest()

from atlas_quant.strategies.filing_momentum_ml.sector_encoding import SectorEncoder
inputs = ProductionRunInputs(
    backtest_config=FilingMomentumBacktestConfig(), periods=periods, universe=bundle.universe,
    benchmark_instrument_id=bundle.benchmark_instrument_id, trading_calendar=calendar,
    sector_encoder=SectorEncoder(), filings_by_instrument=bundle.filings_by_instrument,
    prices_by_instrument=bundle.prices_by_instrument, sector_by_instrument=bundle.sector_by_instrument,
    manifest=manifest,  # checkpoint_root left None -- this notebook never writes a checkpoint file
)
result = run_filing_momentum_production_backtest(inputs)
print("state:", result.state.value)
print("reason:", result.blocked_reason)
for dep in result.missing_dependencies:
    print(" -", dep.name, dep.availability.value)


## Findings

- The orchestration runs the dependency gate first and refuses to proceed past it rather than attempting a partial or faked run. Whether this cell reports `BLOCKED_MISSING_DEPENDENCY` or `COMPLETED` therefore depends on which optional dependencies are installed in the environment you run it in -- with `scikit-learn` present it completes over synthetic data, so the numbers remain meaningless.

## Limitations

- Once scikit-learn/requests are installed and real production data (not this notebook's synthetic bundle) is supplied, re-running this cell would proceed through Stage 3 feature build, Stage 6/7 labeling/training/backtest, Stage 8 performance analysis, and (if `report_options` is set) Stage 9 reporting -- see notebook 09 for a demonstration of that downstream path using an explicitly labeled test fake.

## Update: real-data recovery after the cohort-snapshot correction

This notebook's own synthetic bundle is too small to demonstrate training eligibility. Against Stage 11's real acquired data (518 symbols, 2015-03-31..2024-12-31, 40 shared cohorts), the corrected feature build produces 18,960 successful observations out of 20,720 candidate attempts (91.5%) -- every rejection is genuine data insufficiency ("no fundamental history knowable as of data_cutoff"), never a fiscal/calendar mismatch. Training-quarter eligibility, which was 0 of every requested quarter before this correction, is 32 of 39 candidate target quarters afterward. See `docs/reproducibility_findings.md` for the full real-data diagnostics and the current classification.